# S09 - Fine-Tuning Pre-trained Language Models
## Exercises

These exercises cover the key concepts from Session 09: the pre-train + fine-tune paradigm, contextual embeddings, BERT, GPT, parameter-efficient fine-tuning (LoRA), and running LLMs locally with Ollama.

---
### Exercise 1: Loading a Pre-trained BERT Model (Easy)

As discussed in the slides, BERT (Devlin et al., 2018) is a bidirectional Transformer encoder pre-trained on large unlabeled text using Masked Language Modeling (MLM) and Next Sentence Prediction (NSP). HuggingFace provides easy access to pre-trained models.

**Task:** Load the `bert-base-uncased` model and its tokenizer using HuggingFace's `AutoTokenizer` and `AutoModel`. Tokenize the sentence `"Hello, how are you?"` and print:
- The input IDs (integer indices)
- The decoded tokens (including special tokens [CLS] and [SEP])

In [3]:
from transformers import AutoTokenizer, AutoModel

# 1. Load 'bert-base-uncased' model and tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 2. Tokenize: "Hello, how are you?"
text = "Hello, how are you?"
# The tokenizer returns a dictionary containing 'input_ids', 'token_type_ids', and 'attention_mask'
inputs = tokenizer(text)
input_ids = inputs["input_ids"]

# 3. Print the input IDs and the decoded tokens
print(f"Input IDs: {input_ids}")

# Convert the integer IDs back to string tokens to reveal the [CLS] and [SEP] special tokens
decoded_tokens = tokenizer.convert_ids_to_tokens(input_ids)
print(f"Decoded tokens: {decoded_tokens}")


c:\Users\ybual\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ybual\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ybual\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an adminis

Input IDs: [101, 7592, 1010, 2129, 2024, 2017, 1029, 102]
Decoded tokens: ['[CLS]', 'hello', ',', 'how', 'are', 'you', '?', '[SEP]']


---
### Exercise 2: Contextual Embeddings with BERT (Easy)

Unlike static embeddings (Word2Vec, GloVe) where each word has a single fixed vector, BERT produces contextual embeddings: the same word gets a different vector depending on its surrounding context. The [CLS] token at position 0 is commonly used as a sentence-level representation for classification tasks.

**Task:** Using the model and tokenizer from Exercise 1, obtain the BERT embedding for the sentence `"Natural language processing is fascinating"`. Extract the [CLS] token embedding (position 0 of the last hidden state) and print its shape. The expected shape is `(1, 768)` since BERT-base has a hidden dimension of 768.

In [ ]:
import torch

# Get the [CLS] token embedding for: "Natural language processing is fascinating"
# Print the embedding shape


---
### Exercise 3: Fine-Tuning BERT for Sentiment Classification (Medium)

As covered in the slides, the general fine-tuning framework for text classification is:
1. Initialize with pre-trained BERT weights
2. Add a linear classifier on top of the [CLS] token: `W * h_[CLS] + b`
3. Fine-tune ALL parameters end-to-end (typically 2-4 epochs)

HuggingFace provides `AutoModelForSequenceClassification` which adds the classification head automatically, and the `Trainer` API which handles the training loop.

**Task:** Create a small sentiment dataset with the following samples:
- Positive (label 1): `"I love this!"`, `"Great product"`
- Negative (label 0): `"Terrible experience"`, `"Waste of money"`

Load `bert-base-uncased` for sequence classification with 2 labels. Set up a `Trainer` with:
- 3 training epochs
- Batch size of 2
- Logging every step

You may leave `trainer.train()` commented out to avoid long execution times in class.

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# Create a small dataset
data = {
    'text': ['I love this!', 'Great product', 'Terrible experience', 'Waste of money'],
    'label': [1, 1, 0, 0]
}

# Load model for classification and set up training


---
### Exercise 4: Discriminative Fine-Tuning with Learning Rate Scheduling (Medium)

The slides discuss several important practical techniques for fine-tuning:
- **Discriminative fine-tuning** (Howard & Ruder, 2018 - ULMFiT): use different learning rates for different layers. Lower layers learn general features and should change slowly (small LR), while higher layers and the classifier head are more task-specific and can use a larger LR.
- **Learning rate warmup**: start with a very small LR and gradually increase it during the first 6-10% of training steps, then linearly decay. This prevents large early updates from destabilizing pre-trained weights.
- These techniques help prevent **catastrophic forgetting**, where fine-tuning overwrites the general language knowledge learned during pre-training.

**Task:** Using the model from Exercise 3:
1. Create an AdamW optimizer with two parameter groups:
   - BERT layers with LR = 2e-5
   - Classifier head with LR = 1e-4
2. Set up a linear warmup schedule using `get_linear_schedule_with_warmup` with 10% warmup steps.
3. Print the total training steps and warmup steps.

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Set up optimizer with different learning rates for BERT layers vs classifier head
# Implement warmup schedule


---
### Exercise 5: LoRA - Low-Rank Adaptation for Efficient Fine-Tuning (Hard)

Full fine-tuning updates ALL model parameters, which is expensive for large models. The slides present several Parameter-Efficient Fine-Tuning (PEFT) methods. The most popular is **LoRA** (Hu et al., 2022), which is based on the insight that weight updates during fine-tuning have low intrinsic rank.

Instead of updating a weight matrix W directly, LoRA decomposes the update into two low-rank matrices:

    W' = W + delta_W = W + B * A
    where A has shape (d, r) and B has shape (r, d), with r << d

The original weights W are frozen, and only A and B are trained, drastically reducing the number of trainable parameters (typically ~1% of total parameters).

**Task:** Using the `peft` library:
1. Load a `bert-base-uncased` model for sequence classification
2. Print the total number of parameters before applying LoRA
3. Configure LoRA with rank r=8, alpha=32, dropout=0.1, targeting the `query` and `value` attention matrices
4. Apply LoRA and print the number of trainable parameters and the percentage of total parameters

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType

# Apply LoRA to a pre-trained model
# Compare trainable parameters before and after
